# Task 2 — Weekend Planner Multi-Agent System

Imports from `weekend_planner_task2.py`. This notebook covers every deliverable
requirement: two different branches (both run in the demo), the parallel step,
the critic's loop-back, the HITL breakpoint, long-term memory across two
separate sessions, and using time-travel to debug a real failure.


In [1]:
%pip install -q -r requirements.txt python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

from dotenv import load_dotenv
load_dotenv(".env")
print(os.environ.get("TAVILY_API_KEY")[:8])

assert os.environ.get("GOOGLE_API_KEY")
assert os.environ.get("TAVILY_API_KEY")

from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command

from weekend_planner_task2 import builder

# Same store and same checkpointer for both runs, so long-term memory is
# actually shared across sessions (not just within one thread).
memory = MemorySaver()
long_term_store = InMemoryStore()
graph = builder.compile(checkpointer=memory, store=long_term_store)


tvly-dev


c:\Users\najaf\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. First branch — "go out" (go_out) — single path

Note: it will pause at `approve_and_remember` (HITL) — that's expected, not an error.


In [3]:
config_1 = {"configurable": {"thread_id": "session-1"}}

result = graph.invoke(
    {
        "messages": [HumanMessage(content="طفشانة، ودي أطلع بسيهات")],
        "mood": "bored",
        "city": "Saihat",
        "recommendations": {},
        "revision_count": 0,
    },
    config_1,
)
print(result.get("__interrupt__"))


[Interrupt(value={'question': 'وافقي على هذي التوصية قبل ما نحفظها بالذاكرة الدائمة؟', 'preview': {'weather': {'time': '2026-08-04T16:30', 'interval': 900, 'temperature': 35.1, 'windspeed': 13.0, 'winddirection': 95, 'is_day': 0, 'weathercode': 0}, 'spot_suggestion': [{'url': 'https://alqaheranews.net/news/160949/%D8%A7%D9%84%D8%B3%D8%B9%D9%88%D8%AF%D9%8A%D8%A9-%D8%A7%D8%B9%D8%AA%D8%B1%D8%A7%D8%B6-%D9%88%D8%AA%D8%AF%D9%85%D9%8A%D8%B1-%D9%85%D8%B3%D9%8A%D8%B1%D8%A7%D8%AA-%D9%81%D9%8A-%D8%A7%D9%84%D9%85%D9%86%D8%B7%D9%82%D8%A9-%D8%A7%D9%84%D8%B4%D8%B1%D9%82%D9%8A%D8%A9', 'title': 'السعودية: اعتراض وتدمير 7 مسيّرات في المنطقة الشرقية | القاهرة الاخبارية', 'content': 'أخبار\n\n##### الرئيس المصري وولي عهد السعودية يشددان على احترام سيادة الدول العربية\n\nالشرق الأوسط\n\n##### وزير الخارجية المصري يتوجه للمملكة العربية السعودية\n\nأخبار\n\n##### الدفاعات السعودية تعترض 70 صاروخا و450 مسيرة قبل دخول أجواء المملكة\n\nأخبار\n\n##### الدفاع السعودية: اعتراض وتدمير 3 مُسيّرات في الرياض والشرقية\

In [4]:
# Approve the recommendation so it gets written to long-term memory
result = graph.invoke(Command(resume="موافقة"), config_1)
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

طفشانة، ودي أطلع بسيهات
================================== Ai Message ==================================

[{'type': 'text', 'text': 'إليك تلخيص التوصيات والمعلومات المرفقة:\n\n**باللغة العربية:**\n1. تشير البيانات إلى طقس حار ليلاً بدرجة حرارة تبلغ 35.1 درجة مئوية وسماء صافية مع رياح معتدلة.\n2. بالتزامن مع ذلك، تُظهر التقرير الإخبارية نجاح الدفاعات الجوية السعودية في اعتراض وتدمير عدة طائرات مسيّرة بالمنطقة الشرقية، مع اتخاذ تدابير لوجستية لتأمين سلاسل الإمداد.\n3. يُنصح بالبقاء في أماكن مكيفة لتجنب الحرارة العالية ومتابعة المستجدات الأمنية بانتظام لضمان السلامة.\n\n---\n\n**In English:**\n1. The weather forecast indicates warm night conditions with temperatures reaching 35.1°C, clear skies, and light winds.\n2. Meanwhile, news reports highlight the successful interception of multiple drones by Saudi Air Defense in the Eastern Region, along with measures to secure local supply chains.\n3. It is recommend

## 2. Second branch — "stay home" (stay_home) — parallel step

This time `book_agent` and `media_agent` run at the same time. Try a brand new
thread (`session-2`) — but with the same `long_term_store`, so we can prove
long-term memory works across the two sessions.


In [5]:
config_2 = {"configurable": {"thread_id": "session-2"}}

result = graph.invoke(
    {
        "messages": [HumanMessage(content="طفشانة، ودي اقعد بالبيت بالظهران")],
        "mood": "bored",
        "city": "Dhahran",
        "recommendations": {},
        "revision_count": 0,
    },
    config_2,
)
print(result.get("__interrupt__"))


[Interrupt(value={'question': 'وافقي على هذي التوصية قبل ما نحفظها بالذاكرة الدائمة؟', 'preview': {'weather': {'time': '2026-08-04T16:30', 'interval': 900, 'temperature': 36.3, 'windspeed': 16.1, 'winddirection': 76, 'is_day': 0, 'weathercode': 0}, 'book_agent_error': "HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=10)", 'trending_movie': [{'url': 'https://ar.wikipedia.org/wiki/%D9%82%D8%A7%D8%A6%D9%85%D8%A9_%D8%A3%D8%B9%D9%84%D9%89_%D8%A7%D9%84%D8%A3%D9%81%D9%84%D8%A7%D9%85_%D8%AF%D8%AE%D9%84%D8%A7_%D9%81%D9%8A_%D8%A7%D9%84%D8%B3%D8%B9%D9%88%D8%AF%D9%8A%D8%A9', 'title': 'قائمة أعلى الأفلام دخلا في السعودية - ويكيبيديا', 'content': '| الترتيب | الفيلم | عدد التذاكر | السنة | المصادر |\n ---  --- \n| 01 | فتيان أشقياء: اركب أو مت | 1,700,000 | 2024 |  |\n| 02 | توب غان: مافريك | 1,250,059 | 2022 |  |\n| 03 | سبايدرمان: لا عودة إلى الديار | 966,000 | 2021 |  |\n| 04 | بحبك "بحبك (فيلم)") | 935,595 | 2022 |  |\n| 05 | سطار "سطار (فيلم)") | 903,705 | 2

In [6]:
result = graph.invoke(Command(resume="موافقة"), config_2)
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

طفشانة، ودي اقعد بالبيت بالظهران
================================== Ai Message ==================================

[{'type': 'text', 'text': 'إليك ملخص التوصيات والمعلومات المتاحة باللغتين العربية والإنجليزية:\n\n**بالعربية:**\nالطقس حار الليلة حيث تبلغ درجة الحرارة 36.3° مئوية، مما يجعله وقتاً مثالياً للبقاء في المنزل ومشاهدة الأفلام الأكثر رواجاً في السعودية مثل "Bad Boys: Ride or Die" أو قائمة Top 10 على نتفليكس. نود إحاطتك بأن توصيات الكتب غير متوفرة حالياً بسبب خطأ مؤقت في الاتصال بالمكتبة. نتمنى لك سهرة ممتعة مع قائمة الأفلام المقترحة!\n\n---\n\n**In English:**\nThe weather tonight is warm at 36.3°C, making it an ideal time to stay indoors and watch top-trending movies in Saudi Arabia, such as "Bad Boys: Ride or Die" or popular Netflix hits. Please note that book recommendations are currently unavailable due to a temporary connection error. Enjoy your movie night with the top-ranked film suggestions

## 2b. Extra scenario — "go out for a movie/game" (go_out_media, parallel)

This exercises the cinema/arcade branch, which runs `place_agent` and
`media_agent` in parallel too.


In [7]:
config_2b = {"configurable": {"thread_id": "session-2b"}}

result = graph.invoke(
    {
        "messages": [HumanMessage(content="طفشانة، ودي فيلم بالسينما بالخبر")],
        "mood": "bored",
        "city": "Khobar",
        "recommendations": {},
        "revision_count": 0,
    },
    config_2b,
)
print(result.get("__interrupt__"))


[Interrupt(value={'question': 'وافقي على هذي التوصية قبل ما نحفظها بالذاكرة الدائمة؟', 'preview': {'weather': {'time': '2026-08-04T16:30', 'interval': 900, 'temperature': 35.1, 'windspeed': 14.0, 'winddirection': 74, 'is_day': 0, 'weathercode': 0}, 'book_agent_error': "HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=10)", 'trending_movie': [{'url': 'https://ar.wikipedia.org/wiki/%D9%82%D8%A7%D8%A6%D9%85%D8%A9_%D8%A3%D8%B9%D9%84%D9%89_%D8%A7%D9%84%D8%A3%D9%81%D9%84%D8%A7%D9%85_%D8%AF%D8%AE%D9%84%D8%A7_%D9%81%D9%8A_%D8%A7%D9%84%D8%B3%D8%B9%D9%88%D8%AF%D9%8A%D8%A9', 'title': 'قائمة أعلى الأفلام دخلا في السعودية - ويكيبيديا', 'content': '| الترتيب | الفيلم | عدد التذاكر | السنة | المصادر |\n ---  --- \n| 01 | فتيان أشقياء: اركب أو مت | 1,700,000 | 2024 |  |\n| 02 | توب غان: مافريك | 1,250,059 | 2022 |  |\n| 03 | سبايدرمان: لا عودة إلى الديار | 966,000 | 2021 |  |\n| 04 | بحبك "بحبك (فيلم)") | 935,595 | 2022 |  |\n| 05 | سطار "سطار (فيلم)") | 903,705 | 2

In [8]:
result = graph.invoke(Command(resume="موافقة"), config_2b)
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

طفشانة، ودي فيلم بالسينما بالخبر
================================== Ai Message ==================================

[{'type': 'text', 'text': 'إليك ملخص التوصيات باللغتين العربية والإنجليزية:\n\n**باللغة العربية:**\nنظراً لأن الطقس دافئ ليلاً بدرجة حرارة تبلغ 35.1°م، يُنصح بالاستمتاع بالأنشطة الداخلية ومشاهدة الأفلام الرائجة مثل "فتيان أشقياء: اركب أو مت" أو فيلم "Love in Slow Motion" عبر منصة نتفليكس. كما نود إحاطتك بتعذر تقديم توصيات الكتب حالياً بسبب خطأ مؤقت في الاتصال بالخدمة.\n\n---\n\n**In English:**\nGiven the warm night weather at around 35.1°C, we recommend enjoying indoor entertainment with top trending movies such as "Bad Boys: Ride or Die" or Netflix\'s "Love in Slow Motion." Please note that book recommendations are currently unavailable due to a temporary service connection error.', 'extras': {'signature': 'EsIeCr8eARFNMg/w/JHI8S05ZwRKeEI3/EH6zLGQKPESgKhwtsv4SqAR3g+O5OzRSN7JonfhJW2s+sZ1YwvSg

## 2c. Ambiguous message — should ask instead of guess

No branch keyword at all — the graph should pause with a clarifying question
(via the `clarify` node), not silently pick a branch from the weather alone.


In [9]:
config_2c = {"configurable": {"thread_id": "session-2c"}}

result = graph.invoke(
    {
        "messages": [HumanMessage(content="طفشانة بس")],
        "mood": "bored",
        "city": "Dammam",
        "recommendations": {},
        "revision_count": 0,
    },
    config_2c,
)
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

طفشانة بس
================================== Ai Message ==================================

مو واضح لي وش تفضلين اليوم! الجو الحين 36.6°م. تحبين تطلعين برا (مطعم أو نشاط)، تطلعين لمكان فيه فيلم أو ألعاب (سينما/صالة ألعاب)، أو تقعدين بالبيت (كتاب أو فيلم)؟


## 3. Proving long-term memory across sessions

We read directly from the store (bypassing the graph) — we should see
suggestions saved from both sessions together, even though their `thread_id`s
are completely different.


In [10]:
saved_items = long_term_store.search(("najaf", "suggestions"))
for item in saved_items:
    print(item.key, "->", item.value)


2177822861452493455 -> {'recommendations': {'weather': {'time': '2026-08-04T16:30', 'interval': 900, 'temperature': 35.1, 'windspeed': 13.0, 'winddirection': 95, 'is_day': 0, 'weathercode': 0}, 'spot_suggestion': [{'url': 'https://alqaheranews.net/news/160949/%D8%A7%D9%84%D8%B3%D8%B9%D9%88%D8%AF%D9%8A%D8%A9-%D8%A7%D8%B9%D8%AA%D8%B1%D8%A7%D8%B6-%D9%88%D8%AA%D8%AF%D9%85%D9%8A%D8%B1-%D9%85%D8%B3%D9%8A%D8%B1%D8%A7%D8%AA-%D9%81%D9%8A-%D8%A7%D9%84%D9%85%D9%86%D8%B7%D9%82%D8%A9-%D8%A7%D9%84%D8%B4%D8%B1%D9%82%D9%8A%D8%A9', 'title': 'السعودية: اعتراض وتدمير 7 مسيّرات في المنطقة الشرقية | القاهرة الاخبارية', 'content': 'أخبار\n\n##### الرئيس المصري وولي عهد السعودية يشددان على احترام سيادة الدول العربية\n\nالشرق الأوسط\n\n##### وزير الخارجية المصري يتوجه للمملكة العربية السعودية\n\nأخبار\n\n##### الدفاعات السعودية تعترض 70 صاروخا و450 مسيرة قبل دخول أجواء المملكة\n\nأخبار\n\n##### الدفاع السعودية: اعتراض وتدمير 3 مُسيّرات في الرياض والشرقية\n\nأخبار\n\n##### السعودية تعترض 41 مسيرة في المنطقة ال

## 4. Critic loop-back (optional — force a failure to watch it recover)

To actually see the critic send work back: temporarily break the Tavily key
before running a new path, so `place_agent` or `media_agent` come back empty,
and watch `revision_count` increase and `critic_feedback` explain why — before
it hits the cap (2) and proceeds anyway despite the gap.


In [11]:
_real_key = os.environ["TAVILY_API_KEY"]
os.environ["TAVILY_API_KEY"] = "sk-invalid-broken-key"

config_3 = {"configurable": {"thread_id": "critic-test"}}
result = graph.invoke(
    {
        "messages": [HumanMessage(content="طفشانة، ودي أطلع بالخبر")],
        "mood": "bored",
        "city": "Khobar",
        "recommendations": {},
        "revision_count": 0,
    },
    config_3,
)
print("revision_count:", result.get("revision_count"))
print("critic_feedback:", result.get("critic_feedback"))

os.environ["TAVILY_API_KEY"] = _real_key


revision_count: 2
critic_feedback: ناقص: spot_suggestion — إعادة محاولة


## 5. Time-travel — fixing a real failure


In [12]:
for snap in graph.get_state_history(config_3):
    print(snap.config["configurable"].get("checkpoint_id"), "-> next:", snap.next)


1f190236-cad2-61d6-8005-b92687c2ffc7 -> next: ('approve_and_remember',)
1f190236-cacc-6e23-8004-6aacec51d068 -> next: ('critic',)
1f190236-c7fd-6982-8003-d50fd0f9ed55 -> next: ('place_agent',)
1f190236-c7e1-67c9-8002-ac9fc276df06 -> next: ('critic',)
1f190236-bef6-6f9a-8001-50a9aa108a5a -> next: ('place_agent',)
1f190236-b8c1-609b-8000-e456ee98afdb -> next: ('router',)
1f190236-b855-6a8b-bfff-3e79b7ce6ff4 -> next: ('__start__',)


In [14]:
replay_config = {"configurable": {"thread_id": "critic-test", "checkpoint_id": "1f190236-bef6-6f9a-8001-50a9aa108a5a"}}
result = graph.invoke(None, replay_config)
print("critic_feedback:", result.get("critic_feedback"))
print("spot_suggestion:", result.get("recommendations", {}).get("spot_suggestion"))

critic_feedback: موافق عليه
spot_suggestion: [{'url': 'https://saudipedia.com/article/12279/%D8%A7%D9%82%D8%AA%D8%B5%D8%A7%D8%AF-%D9%88%D8%A3%D8%B9%D9%85%D8%A7%D9%84/%D8%B4%D8%B1%D9%83%D8%A7%D8%AA/%D8%B4%D8%B1%D9%83%D8%A9-%D8%A3%D8%B3%D9%85%D9%86%D8%AA-%D8%A7%D9%84%D9%85%D9%86%D8%B7%D9%82%D8%A9-%D8%A7%D9%84%D8%B4%D8%B1%D9%82%D9%8A%D8%A9', 'title': 'سعوديبيديا - شركة أسمنت المنطقة الشرقية', 'content': '# شركة أسمنت المنطقة الشرقية\n\nمقالة\n\n1 د\n\n09/05/2023\n\nشركات\n\ncopied\n\nشركة أسمنت المنطقة الشرقية، هي شركة سعودية مُساهمة ومدرجة في السوق المالية السعودية "تداول السعودية"، تأسست في  1403هـ/ 1983م، ومقرّها الرئيس في مدينة الخبر. وتعمل في صناعة الأسمنت والكلنكر.\n\nيقع مصنع الشركة في منطقة الخرسانية على طريق أبو حدرية السريع على بعد (145) كم شمال مدينة الدمام ، حيث تتوفر المواد اللازمة لصناعة الأسمنت. [...] يتمثّل نشاط شركة أسمنت بشكلٍ أساسي في صناعة الأسمنت والكلنكر لتلبية احتياجات السوق المحلية والخليجية، ويقع مصنـع الشركة في منطقة الخرسانية على طريق أبو حدرية السريع على بعد 14

## 6. Notes for submission

- **3+ nodes with different jobs and tools:** `router` (weather tool),
  `place_agent` (search_local_spot), `book_agent` (get_book_recommendation),
  `media_agent` (get_trending_movie_or_game), `critic` (no tool, logic-only
  validation), `approve_and_remember` (store + interrupt), `respond` (llm,
  no tools).

- **Conditional routing with both branches run in the demo:** proven above in
  sections 1 and 2 (and 2b for the extra go_out_media branch, 2c for the
  ambiguous "ask instead of guess" case).

- **Parallel step with a reducer:** `stay_home_dispatch` and `go_out_dispatch`
  each run two specialists at the same time; results merge via
  `merge_recommendations` without overwriting each other.

- **Critic loop-back with a hard cap:** proven twice — once deliberately in
  section 4 (`revision_count` stops at 2), and once from a real unplanned
  failure in section 2: `book_agent` hit an actual `Read timed out` error
  against Open Library, the critic detected the missing `book_suggestion`,
  retried, hit the cap, and proceeded anyway with a partial result instead of
  crashing.

- **HITL breakpoint before something irreversible:** the pause before
  `approve_and_remember` (writing to permanent memory) in sections 1, 2, 2b.

- **Long-term memory across two sessions:** proven in section 3 — recommendations
  were saved under the same store across different `thread_id`s, confirming
  the store persists independently of any single conversation thread.

- **Time-travel used to debug a real failure:** replayed from checkpoint
  `...bef6...` (right before the first `place_agent` call). The first replay
  attempt failed again with the same "missing spot_suggestion" error even
  after the correct Tavily key had been restored in the environment. Root
  cause: `_get_tavily_client()` cached the client object in a module-level
  global the first time it was called during the broken-key test, so it had
  already baked in the invalid key — restoring `os.environ` afterward had no
  effect on the already-cached client. This was a **tool-layer bug**, not a
  model or planning failure. **Fix applied:** `_get_tavily_client()` now
  tracks which key it last built the client with and rebuilds automatically
  if `os.environ["TAVILY_API_KEY"]` changes. After the fix, replaying from
  the same point succeeded: `critic_feedback` came back `"موافق عليه"` and
  `spot_suggestion` returned real search results on the first attempt — no
  further retries needed. (Side note: the returned results were only loosely
  related to the actual restaurant/activity query, which suggests the
  `search_local_spot` query string could be made more specific in a future
  iteration — noted here as an honest observation, not something we fixed.)

- **Why we split the work this way:** each specialist owns exactly one
  external service, so a failure in one (e.g. `book_agent` timing out
  against Open Library) never blocks the others — `media_agent` still
  returned a trending movie in the same run. Keeping `router` and `critic`
  free of any tool call means the only two jobs that can silently fail —
  deciding the branch and validating the result — stay simple enough to
  reason about and test in isolation.
